In [ ]:
%env WORKDIR=/tmp/vault-pr
%env VAULT_K8S_NAMESPACE=vaultpr
%env VAULT_HELM_RELEASE_NAME=vaultpr
%env VAULT_SERVICE_NAME=vaultpr-internal 
%env K8S_CLUSTER_NAME=cluster.local

In [ ]:
! mkdir -p $WORKDIR

In [ ]:
! minikube status -p PR || minikube start -p PR --force


In [ ]:
! minikube update-context -p PR


In [ ]:
%%bash
export VAULT_K8S_NAMESPACE="vaultpr"
export VAULT_HELM_RELEASE_NAME="vaultpr"
export VAULT_SERVICE_NAME="vaultpr-internal"
export K8S_CLUSTER_NAME="cluster.local"

openssl genrsa -out ${WORKDIR}/vault.key 2048

cat > ${WORKDIR}/vaultpr-csr.conf <<EOF
[req]
default_bits = 2048
prompt = no
encrypt_key = yes
default_md = sha256
distinguished_name = kubelet_serving
req_extensions = v3_req
[ kubelet_serving ]
O = system:nodes
CN = system:node:*.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
[ v3_req ]
basicConstraints = CA:FALSE
keyUsage = nonRepudiation, digitalSignature, keyEncipherment, dataEncipherment
extendedKeyUsage = serverAuth, clientAuth
subjectAltName = @alt_names
[alt_names]
DNS.1 = *.${VAULT_SERVICE_NAME}
DNS.2 = *.${VAULT_SERVICE_NAME}.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
DNS.3 = *.${VAULT_HELM_RELEASE_NAME}
DNS.4 = *.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
IP.1 = 127.0.0.1
EOF

openssl req -new -key ${WORKDIR}/vault.key -out ${WORKDIR}/vaultpr.csr -config ${WORKDIR}/vaultpr-csr.conf


cat > ${WORKDIR}/csrpr.yaml <<EOF
apiVersion: certificates.k8s.io/v1
kind: CertificateSigningRequest
metadata:
   name: vaultpr.svc
spec:
   signerName: kubernetes.io/kubelet-serving
   expirationSeconds: 8640000
   request: $(cat ${WORKDIR}/vaultpr.csr|base64|tr -d '\n')
   usages:
   - digital signature
   - key encipherment
   - server auth
EOF

kubectl --context=PR delete csr vaultpr.svc --ignore-not-found
kubectl --context=PR create -f ${WORKDIR}/csrpr.yaml

In [ ]:
%%bash
kubectl --context=PR certificate approve vaultpr.svc

In [ ]:
%%bash
kubectl --context=PR config view \
--raw \
--minify \
--flatten \
-o jsonpath='{.clusters[].cluster.certificate-authority-data}' \
| base64 -d > ${WORKDIR}/vault.ca

In [ ]:
%%bash
kubectl --context=PR get csr vaultpr.svc

In [ ]:
%%bash
kubectl --context=PR get csr vaultpr.svc -o jsonpath='{.status.certificate}' | openssl base64 -d -A -out ${WORKDIR}/vaultpr.crt

In [ ]:
%%bash


kubectl --context=PR create namespace $VAULT_K8S_NAMESPACE --dry-run=client -o yaml | kubectl --context=PR apply -f -

kubectl --context=PR create secret generic vault-ha-tls \
   -n $VAULT_K8S_NAMESPACE \
   --from-file=vault.key=${WORKDIR}/vault.key \
   --from-file=vault.crt=${WORKDIR}/vaultpr.crt \
   --from-file=vault.ca=${WORKDIR}/vault.ca \
   --dry-run=client -o yaml | kubectl --context=PR apply -f -


In [ ]:
%%bash 
secret=$(cat vault.hclic)
kubectl --context=PR create secret generic vault-ent-license \
  --from-literal="license=${secret}" \
  -n $VAULT_K8S_NAMESPACE \
  --dry-run=client -o yaml | kubectl --context=PR apply -f -

In [ ]:
%%bash
cat > ${WORKDIR}/overrides_pr.yaml <<EOF
global:
   enabled: true
   tlsDisable: false # Disabling TLS to avoid issues when connecting to Vault via port forwarding
injector:
   enabled: false
server:
# config.yaml
   image:
      repository: hashicorp/vault-enterprise
      tag: 2.0.3-ent
   enterpriseLicense:
      secretName: vault-ent-license
   extraEnvironmentVars:
      VAULT_CACERT: /vault/userconfig/vault-ha-tls/vault.ca
      VAULT_TLSCERT: /vault/userconfig/vault-ha-tls/vault.crt
      VAULT_TLSKEY: /vault/userconfig/vault-ha-tls/vault.key
      VAULT_CLIENT_CERT: /vault/userconfig/vault-ha-tls/vault.crt
      VAULT_CLIENT_KEY: /vault/userconfig/vault-ha-tls/vault.key
   volumes:
      - name: userconfig-vault-ha-tls
        secret:
         defaultMode: 420
         secretName: vault-ha-tls
   volumeMounts:
      - mountPath: /vault/userconfig/vault-ha-tls
        name: userconfig-vault-ha-tls
        readOnly: true
   standalone:
      enabled: false
   affinity: ""
   ha:
      enabled: true
      replicas: 3
      raft:
         enabled: true
         setNodeId: true
         config: |
            ui = true
            cluster_name = "vault-perf-secondary"
            listener "tcp" {
               tls_disable = 0 # Disabling TLS to avoid issues when connecting to Vault via port forwarding
               address = "[::]:8200"
               cluster_address = "[::]:8201"
               tls_cert_file = "/vault/userconfig/vault-ha-tls/vault.crt"
               tls_key_file  = "/vault/userconfig/vault-ha-tls/vault.key"
               tls_client_ca_file = "/vault/userconfig/vault-ha-tls/vault.ca"
            }
            storage "raft" {
               path = "/vault/data"
            
               retry_join {
                  auto_join             = "provider=k8s namespace=vaultpr label_selector=\"component=server,app.kubernetes.io/name=vault\""
                  auto_join_scheme      = "https"
                  leader_ca_cert_file   = "/vault/userconfig/vault-ha-tls/vault.ca"
                  leader_tls_servername = "vaultpr-0.vaultpr-internal" #Tiene que matchear una SAN del certificado
               }
            
            }
            disable_mlock = true
            service_registration "kubernetes" {}
EOF

In [ ]:
%%bash
helm --kube-context PR upgrade --install -n $VAULT_K8S_NAMESPACE $VAULT_HELM_RELEASE_NAME hashicorp/vault --version 0.34.0 -f ${WORKDIR}/overrides_pr.yaml --wait --timeout 10m

In [ ]:
%%bash
kubectl --context=PR get events -n vaultpr

In [ ]:
%%bash
kubectl --context=PR get pods -n vaultpr

In [ ]:
%%bash
set -euo pipefail
umask 077

if kubectl --context=PR exec -n "${VAULT_K8S_NAMESPACE}" vaultpr-0 -- \
  env VAULT_SKIP_VERIFY=true vault status -format=json | jq -e '.initialized' >/dev/null; then
  echo "Vault PR is already initialized."
else
  kubectl --context=PR exec -n "${VAULT_K8S_NAMESPACE}" vaultpr-0 -- \
    env VAULT_SKIP_VERIFY=true vault operator init \
      -key-shares=1 \
      -key-threshold=1 \
      -format=json > "${WORKDIR}/clusterpr-keys.json"
  chmod 600 "${WORKDIR}/clusterpr-keys.json"
fi


In [ ]:
%%bash
set -euo pipefail

PR_UNSEAL_KEY=$(jq -r '.unseal_keys_b64[0]' "${WORKDIR}/clusterpr-keys.json")
for pod in vaultpr-0 vaultpr-1 vaultpr-2; do
  for attempt in $(seq 1 60); do
    initialized=$(kubectl --context=PR exec -n "${VAULT_K8S_NAMESPACE}" "$pod" -- \
      env VAULT_SKIP_VERIFY=true vault status -format=json 2>/dev/null | \
      jq -r '.initialized' || echo false)
    [[ "$initialized" == "true" ]] && break
    [[ "$attempt" == "60" ]] && { echo "$pod did not initialize" >&2; exit 1; }
    sleep 3
  done
  kubectl --context=PR exec -n "${VAULT_K8S_NAMESPACE}" "$pod" -- \
    env VAULT_SKIP_VERIFY=true vault operator unseal "$PR_UNSEAL_KEY" >/dev/null
done
unset PR_UNSEAL_KEY


In [ ]:
%%bash
kubectl --context=PR exec -n $VAULT_K8S_NAMESPACE -ti vaultpr-0 -- vault status

In [ ]:
%%bash
kubectl --context=PR exec -n $VAULT_K8S_NAMESPACE -ti vaultpr-0 --  vault license inspect

## Persistir el bootstrap local

El token raíz y la clave Shamir inicial solo se utilizan para activar el secundario. Tras activar la replicación, la clave válida será `VAULT_RECOVERY_KEY` del primario.


In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="${REPO_ROOT:-$(pwd)}"
ENV_FILE="${REPO_ROOT}/.env"
export ENV_FILE
export VAULT_PR_BOOTSTRAP_TOKEN=$(jq -r '.root_token' "${WORKDIR}/clusterpr-keys.json")
export VAULT_PR_INITIAL_UNSEAL_KEY=$(jq -r '.unseal_keys_b64[0]' "${WORKDIR}/clusterpr-keys.json")
export VAULT_PR_CACERT="${WORKDIR}/vault.ca"
umask 077
python3 - <<'PY'
import os
from pathlib import Path

path = Path(os.environ["ENV_FILE"])
updates = {
    "VAULT_PR_BOOTSTRAP_TOKEN": os.environ["VAULT_PR_BOOTSTRAP_TOKEN"],
    "VAULT_PR_INITIAL_UNSEAL_KEY": os.environ["VAULT_PR_INITIAL_UNSEAL_KEY"],
    "VAULT_PR_CACERT": os.environ["VAULT_PR_CACERT"],
}
lines = path.read_text().splitlines() if path.exists() else []
seen, result = set(), []
for line in lines:
    key = line.split("=", 1)[0].strip() if "=" in line else ""
    if key in updates:
        result.append(f"{key}={updates[key]}")
        seen.add(key)
    else:
        result.append(line)
for key, value in updates.items():
    if key not in seen:
        result.append(f"{key}={value}")
path.write_text("\n".join(result).rstrip() + "\n")
path.chmod(0o600)
PY
unset VAULT_PR_BOOTSTRAP_TOKEN VAULT_PR_INITIAL_UNSEAL_KEY
echo "Saved PR bootstrap material in the ignored .env file."
